# AP2 — Auditoria de modelos probabilísticos
## Trilhas Binomial, Poisson e Normal — itens 1 a 4

**Integrantes:** preencher  
**Turma:** preencher  
**Professor(a):** preencher

**Pergunta geral:** os modelos probabilísticos candidatos são compatíveis com as variáveis selecionadas do conjunto Ames Housing?

Este trabalho segue os quatro itens solicitados: delimitação, compatibilidade inicial, descrição empírica e definição do modelo com seus parâmetros. **Não inclui os itens 5, 6 e 7.**

| Trilha | Variável da base | Variável aleatória estudada |
|---|---|---|
| Binomial | `Lot Shape` | Número de terrenos regulares em 20 seleções |
| Poisson | `Fireplaces` | Número de lareiras por imóvel |
| Normal | `SalePrice` | Preço de venda por imóvel, em dólares (US$) |

**Atenção:** essas são as variáveis adotadas neste notebook. Confirme com o professor se houve atribuição de variáveis específicas para seu grupo.

## Carregamento da base da AP1

Usaremos exclusivamente o arquivo `data/processed/AmesHousing.csv`, preparado na AP1. Não faremos novas limpezas ou transformações gerais. As células abaixo verificam apenas as três colunas necessárias. Execute o notebook na raiz do projeto que contém a pasta `data/processed`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display, Markdown

arquivo = Path('data/processed/AmesHousing.csv')
if not arquivo.exists():
    raise FileNotFoundError(
        'Arquivo não encontrado: data/processed/AmesHousing.csv. '
        'Copie para essa pasta o CSV gerado na AP1 e execute novamente.'
    )

df = pd.read_csv(arquivo)
colunas = ['Lot Shape', 'Fireplaces', 'SalePrice']
for coluna in colunas:
    if coluna not in df.columns:
        raise ValueError(f'A coluna {coluna} não foi encontrada no CSV.')

print(f'Base carregada: {len(df)} imóveis e {len(df.columns)} colunas.')
display(pd.DataFrame({
    'Coluna': colunas,
    'Válidos': [df[c].notna().sum() for c in colunas],
    'Ausentes': [df[c].isna().sum() for c in colunas]
}))

# 1. Trilha Binomial — Terreno regular

## Item 1 — Delimitação da variável aleatória

- **Unidade de análise:** cada imóvel registrado na base tratada.
- **Característica observada:** formato do terreno (`Lot Shape`).
- **Evento de sucesso:** `Lot Shape == 'Reg'`, ou seja, terreno regular. Os outros formatos válidos são fracassos.
- **Variável indicadora:** $Y=1$ quando o terreno é regular e $Y=0$ caso contrário; é discreta, com valores possíveis $\{0,1\}$.
- **Variável de interesse:** $X$ = quantidade de terrenos regulares em **20 seleções** de imóveis. É discreta e tem suporte teórico $\{0,1,2,\ldots,20\}$.
- **Pergunta:** o modelo Binomial descreve a quantidade de terrenos regulares em 20 seleções independentes, com a mesma probabilidade de sucesso?

## Item 2 — Compatibilidade inicial

| Critério | Análise |
|---|---|
| Tipo | Compatível: cada seleção resulta em sucesso ou fracasso. |
| Suporte | Compatível: a contagem total varia entre 0 e 20. |
| Mecanismo | Precisamos de 20 tentativas, com dois resultados em cada uma. |
| Probabilidade | O modelo supõe uma mesma probabilidade $p$ em todas as seleções. |
| Independência | Seleções uniformes **com reposição** permitem a hipótese de independência. Sem reposição, a independência não é exata. |

Assim, a Binomial faz sentido para o experimento de 20 seleções com reposição; isso não afirma que a formação dos imóveis no mercado seja independente.

## Item 3 — Descrição empírica

Primeiro contamos quantos terrenos são regulares e quantos têm outros formatos. A média da indicadora $Y$ representa a proporção de terrenos regulares; sua variância descreve a dispersão de respostas 0 ou 1. Para descrever totais de 20 imóveis com dados reais, também organizamos os registros em blocos completos de 20, em ordem fixa, **sem simulação**.

In [ ]:
formatos = df['Lot Shape'].dropna()
validas = {'Reg', 'IR1', 'IR2', 'IR3'}
if not set(formatos.unique()).issubset(validas):
    raise ValueError('Categorias inesperadas em Lot Shape; confira a base da AP1.')

indicadora = (formatos == 'Reg').astype(int)
p = indicadora.mean()
print('Observações válidas:', len(formatos))
print('Ausentes:', df['Lot Shape'].isna().sum())

tabela_bin = pd.DataFrame({
    'Resultado': ['Regular', 'Outro formato'],
    'Frequência': [indicadora.sum(), (1 - indicadora).sum()],
    'Frequência relativa': [p, 1 - p]
})
display(tabela_bin)
print(f'Média da indicadora Y: {indicadora.mean():.4f}')
print(f'Variância observada de Y: {indicadora.var(ddof=0):.4f}')

# A ordem registrada no CSV forma blocos apenas para descrever totais reais de 20.
n = 20
quantidade_blocos = len(indicadora) // n
if quantidade_blocos == 0:
    raise ValueError('A base precisa ter pelo menos 20 formatos válidos.')
blocos = indicadora.to_numpy()[:quantidade_blocos * n].reshape(-1, n).sum(axis=1)
print(f'Blocos completos de 20 imóveis: {quantidade_blocos}')
print(f'Suporte observado nos blocos: {sorted(np.unique(blocos).tolist())}')
print(f'Média dos totais observados: {blocos.mean():.4f}')
print(f'Variância dos totais observados: {blocos.var(ddof=0):.4f}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(tabela_bin['Resultado'], tabela_bin['Frequência relativa'], color=['#277da1', '#90be6d'])
ax.set(title='Formato do terreno — frequência relativa', xlabel='Resultado', ylabel='Proporção de imóveis')
ax.set_ylim(0, 1)
plt.show()

display(Markdown(
    f'**Interpretação:** {p:.1%} dos imóveis com formato informado apresentam terreno regular. '
    f'Nos blocos reais de 20 imóveis, o número de terrenos regulares variou entre '
    f'**{blocos.min()} e {blocos.max()}**, com média **{blocos.mean():.2f}**. '
    'Esses blocos são uma descrição dos dados, não ensaios independentes com reposição.'
))

## Item 4 — Modelo candidato e parâmetros

Propomos $X\sim\operatorname{Binomial}(20,p)$, em que $p$ é a proporção observada de terrenos regulares entre os formatos válidos.

$$E(X)=20p \qquad \operatorname{Var}(X)=20p(1-p)$$

A esperança informa quantos terrenos regulares seriam esperados em cada grupo de 20 seleções com reposição. A variância informa a dispersão teórica dessa contagem, medida em terrenos ao quadrado.

In [ ]:
media_binomial = 20 * p
variancia_binomial = 20 * p * (1 - p)
print(f'Parâmetros: n = 20; p = {p:.4f}')
print(f'Esperança teórica: E(X) = {media_binomial:.4f} terrenos regulares')
print(f'Variância teórica: Var(X) = {variancia_binomial:.4f} terrenos²')

# 2. Trilha Poisson — Número de lareiras

## Item 1 — Delimitação da variável aleatória

- **Unidade de análise:** um imóvel da base tratada.
- **Variável:** $X$ = número de lareiras (`Fireplaces`) existentes em um imóvel.
- **Tipo:** quantitativa discreta, pois é uma contagem inteira.
- **Valores possíveis teoricamente:** $\{0,1,2,3,\ldots\}$ na Poisson; o suporte **observado** será mostrado abaixo.
- **Pergunta:** uma distribuição de Poisson pode representar o número de lareiras por imóvel?

## Item 2 — Compatibilidade inicial

| Critério | Análise |
|---|---|
| Tipo | Compatível: número de lareiras é contagem discreta não negativa. |
| Suporte | A Poisson admite qualquer inteiro não negativo; imóveis têm limite físico de lareiras. |
| Mecanismo | A Poisson costuma contar ocorrências em intervalos ou exposições definidos; aqui contamos elementos de um imóvel. |
| Taxa | Supõe uma taxa média aproximadamente constante, embora imóveis diferentes possam ter características distintas. |
| Independência | Não há garantia de que a presença de cada lareira surja de eventos independentes. |

O tipo numérico é compatível, mas o mecanismo e os limites naturais exigem cautela.

## Item 3 — Descrição empírica

Vamos apresentar as frequências de 0, 1, 2 etc. lareiras, além da média e da variância observadas. Uma concentração de zeros ou uma variância distante da média é relevante para essa investigação.

In [ ]:
lareiras = df['Fireplaces'].dropna()
if not pd.api.types.is_numeric_dtype(lareiras):
    raise ValueError('Fireplaces deveria ser numérica.')
if not (np.isfinite(lareiras).all() and (lareiras >= 0).all()
        and (lareiras == np.floor(lareiras)).all()):
    raise ValueError('Fireplaces deve conter números inteiros não negativos.')

freq_pois = lareiras.astype(int).value_counts().sort_index()
tabela_pois = pd.DataFrame({
    'Frequência': freq_pois,
    'Frequência relativa': freq_pois / len(lareiras)
})
display(tabela_pois)
media_lareiras = lareiras.mean()
var_lareiras = lareiras.var(ddof=0)
print('Observações válidas:', len(lareiras))
print('Ausentes:', df['Fireplaces'].isna().sum())
print('Suporte observado:', sorted(lareiras.astype(int).unique().tolist()))
print(f'Média observada: {media_lareiras:.4f} lareiras/imóvel')
print(f'Variância observada: {var_lareiras:.4f} lareiras²')

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(tabela_pois.index, tabela_pois['Frequência relativa'], color='#577590')
ax.set(title='Número de lareiras por imóvel', xlabel='Número de lareiras', ylabel='Frequência relativa')
ax.set_xticks(tabela_pois.index)
plt.show()

display(Markdown(
    f'**Interpretação:** a proporção de imóveis sem lareira é '
    f'**{(lareiras == 0).mean():.1%}**. A média observada é **{media_lareiras:.2f}** '
    f'e a variância é **{var_lareiras:.2f}**. '
    'O gráfico revela como a contagem se concentra nos valores inteiros observados.'
))

## Item 4 — Modelo candidato e parâmetros

O modelo candidato é $X\sim\operatorname{Poisson}(\lambda)$. Definimos o parâmetro $\lambda$ como a **média observada** de lareiras por imóvel.

$$E(X)=\lambda \qquad \operatorname{Var}(X)=\lambda$$

A Poisson prevê que sua média e sua variância sejam iguais. Compararemos essa propriedade com a variância observada sem tratar uma eventual semelhança como prova de adequação.

In [ ]:
lamb = media_lareiras
print(f'Parâmetro lambda: {lamb:.4f} lareiras/imóvel')
print(f'Esperança teórica: E(X) = {lamb:.4f} lareiras/imóvel')
print(f'Variância teórica: Var(X) = {lamb:.4f} lareiras²')
print(f'Variância observada: {var_lareiras:.4f} lareiras²')
print(f'Diferença entre variâncias observada e teórica: {var_lareiras - lamb:.4f}')

# 3. Trilha Normal — Preço de venda

## Item 1 — Delimitação da variável aleatória

- **Unidade de análise:** um imóvel vendido na base tratada.
- **Variável:** $X$ = preço de venda do imóvel (`SalePrice`), em dólares (US$).
- **Tipo:** quantitativa tratada como contínua para modelagem (embora os preços sejam registrados com arredondamento).
- **Valores possíveis no contexto:** preços positivos. Na distribuição Normal, o suporte matemático é toda a reta real $(-\infty,+\infty)$.
- **Pergunta:** o modelo Normal é uma aproximação razoável para a distribuição dos preços de venda dos imóveis analisados?

## Item 2 — Compatibilidade inicial

| Critério | Análise |
|---|---|
| Tipo | Compatível como aproximação: o preço é uma medida quantitativa. |
| Suporte | A Normal admite valores negativos, que não fazem sentido para preços de venda. |
| Formato | A Normal é simétrica, mas preços imobiliários podem apresentar assimetria à direita. |
| Caudas | Preços muito altos podem causar desvios importantes nas caudas. |
| Parâmetros | A média $\mu$ determina o centro; o desvio-padrão $\sigma$ determina a dispersão. |

Um histograma e um gráfico Q–Q ajudarão a examinar a forma empírica; não se pressupõe normalidade automaticamente.

## Item 3 — Descrição empírica

Calcularemos quantidade de observações, ausências, medidas descritivas, média, variância e desvio-padrão. O **histograma** mostra a forma da distribuição e o **Q–Q plot** compara os quantis observados com os de uma Normal. Pontos próximos da reta no centro, mas afastados nas pontas, sugerem diferenças nas caudas.

In [ ]:
precos = df['SalePrice'].dropna()
if not pd.api.types.is_numeric_dtype(precos) or not np.isfinite(precos).all():
    raise ValueError('SalePrice deve ser numérica e conter valores finitos.')
if (precos <= 0).any() or len(precos) < 3:
    raise ValueError('SalePrice precisa ter preços positivos e pelo menos três registros.')

mu = precos.mean()
variancia_preco = precos.var(ddof=0)
sigma = precos.std(ddof=0)
print('Observações válidas:', len(precos))
print('Ausentes:', df['SalePrice'].isna().sum())
display(precos.describe().to_frame('SalePrice (US$)'))
print(f'Média observada: US$ {mu:,.2f}')
print(f'Variância observada: {variancia_preco:,.2f} US$²')
print(f'Desvio-padrão observado: US$ {sigma:,.2f}')
print(f'Assimetria observada: {precos.skew():.3f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(precos, bins=30, edgecolor='white', color='#43aa8b')
axes[0].set(title='Histograma dos preços de venda', xlabel='Preço de venda (US$)', ylabel='Número de imóveis')
stats.probplot(precos, dist='norm', plot=axes[1])
axes[1].set(title='Q–Q plot dos preços de venda', xlabel='Quantis teóricos da Normal', ylabel='Quantis observados (US$)')
plt.tight_layout()
plt.show()

assimetria = precos.skew()
texto_assimetria = ('assimetria à direita' if assimetria > 0.5
                   else 'assimetria à esquerda' if assimetria < -0.5
                   else 'assimetria pouco acentuada')
display(Markdown(
    f'**Interpretação:** o preço médio é **US$ {mu:,.2f}** e o desvio-padrão é '
    f'**US$ {sigma:,.2f}**. A assimetria calculada é **{assimetria:.2f}**, '
    f'caracterizando {texto_assimetria}. '
    'No Q–Q plot, observe especialmente se as extremidades se afastam da linha de referência: '
    'isso sinaliza caudas ou valores extremos diferentes dos esperados sob uma Normal.'
))

## Item 4 — Modelo candidato e parâmetros

Propomos $X\sim\operatorname{Normal}(\mu,\sigma^2)$, em que:

- $\mu$ é a média dos preços observados;
- $\sigma$ é o desvio-padrão observado;
- $\sigma^2$ é a variância observada.

$$E(X)=\mu \qquad \operatorname{Var}(X)=\sigma^2$$

A esperança é o preço central previsto pelo modelo e a variância quantifica a dispersão dos preços ao redor dessa média. Esses parâmetros são calculados apenas para descrever a base, sem inferir características de outros mercados.

In [ ]:
print(f'Modelo: X ~ Normal(mu={mu:,.2f}, sigma²={variancia_preco:,.2f})')
print(f'Esperança teórica: E(X) = US$ {mu:,.2f}')
print(f'Variância teórica: Var(X) = {variancia_preco:,.2f} US$²')
print(f'Desvio-padrão: sigma = US$ {sigma:,.2f}')

---
**Fim do trabalho solicitado:** foram desenvolvidos os itens **1 a 4** das três trilhas. Os parâmetros foram calibrados pela base observada; os itens de simulação, auditoria final dos pressupostos e parecer conclusivo (5 a 7) não fazem parte desta entrega.